> **한눈에 보기**
> - Pydantic 공식 문서 예제: 모델 만들기 → 자동 변환 → 검증 실패 → 출력 옵션
> - `PositiveInt`, `Literal`, `Annotated[float, Gt(0)]`처럼 **타입 힌트만으로 값의 조건**을 정한다
> - 실패하면 `ValidationError`, `e.errors()`로 어느 필드가 왜 틀렸는지 한 번에 본다
> - `model_dump()` 옵션: `exclude_unset`, `exclude`, `mode='json'`, `exclude_defaults`
> - 정리 문서: [Pydantic.md](Pydantic.md)

### 1. 기본 모델과 자동 변환

In [7]:
# PositiveInt: 양의 정수 / Literal: 정해진 값만 / Annotated + Gt(0): 0보다 큰 값
# 모듈 import
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError
from typing import Annotated, Literal 
from annotated_types import Gt

In [5]:
# signup_ts: datetime | None 은 기본값이 없어서, None이라도 '반드시' 넣어야 하는 필드
# tastes: 키는 문자열, 값은 양의 정수인 dict
# 클래스 선언부

class User(BaseModel):
    id: int 
    name: str = 'John Doe' 
    signup_ts: datetime | None 
    tastes: dict[str, PositiveInt]


In [ ]:
# 일부러 타입이 섞인 데이터: 날짜가 문자열, b'cheese'는 bytes 키, '1'은 문자열 숫자
# 데이터
external_data = {
    'id': 123,
    'signup_ts': '2019-06-01 12:22',
    'tastes': {
        'wine': 9,
        b'cheese': 7,
        'cabbage': '1',
    },
}

In [ ]:
# **external_data: dict를 키워드 인자로 풀어서 전달 (id=123, signup_ts=..., tastes=...)
# user 오브젝트 생성
user = User(**external_data)

In [10]:
# 변환 결과 확인: 문자열 -> datetime, b'cheese' -> 'cheese', '1' -> 1
print(user.id)
print(user.signup_ts)
print(user.tastes)

123
2019-06-01 12:22:00
{'wine': 9, 'cheese': 7, 'cabbage': 1}


### 2. 검증 실패: ValidationError

In [2]:
# 검증에 실패할 데이터: id는 숫자가 아니고, signup_ts는 빠져 있다
external_data2 = {
    'id': 'not an int',
    'tastes': {}
}

In [6]:
# e.errors(): 실패한 항목 목록 (type=오류 종류, loc=어느 필드, msg=이유)
try:
    User(**external_data2)
except ValidationError as e:
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


**📝 정리** — 틀린 곳이 여러 개여도 **한 번에 모두** 알려 준다. 여기서는 `id` 형식 오류(`int_parsing`)와 `signup_ts` 누락(`missing`) 두 개.

### 3. Literal, Annotated로 값 제한하기

In [8]:
# Literal['red', 'green']: 이 두 값만 허용
# Annotated[float, Gt(0)]: 0보다 큰 실수
# bazam: dict 안에 list 안에 tuple - 복잡한 구조도 타입 힌트로 검증
class Fruit(BaseModel):
    name: str 
    color: Literal['red', 'green']
    weight: Annotated[float, Gt(0)]
    bazam: dict[str, list[tuple[int, bool, float]]]

In [9]:
# 모든 조건을 만족하므로 정상 생성
print(
    Fruit(
        name='Apple',
        color='red',
        weight=4.2,
        bazam={'foobar': [(1, True, 0.1)],}
    )
)

name='Apple' color='red' weight=4.2 bazam={'foobar': [(1, True, 0.1)]}


### 4. model_dump 출력 옵션

In [10]:
# where: bytes -> 문자열 'home'을 넣으면 b'home'으로 변환
# why: 기본값 'No idea'
class Meeting(BaseModel):
    when: datetime 
    where: bytes 
    why: str='No idea'

In [11]:
# why는 넣지 않음 (기본값 사용)
m = Meeting(when='2020-01-01T12:00', where='home')

In [12]:
# exclude_unset=True: 직접 넣은 필드(when, where)만
print(m.model_dump(exclude_unset=True))

{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}


exclude_unset은 설정하지 않은 속성들은 출력하지 않음
model_dump 메서드는 데이터를 딕셔너리 형태로 출력

In [13]:
# exclude: 뺄 필드 지정 / mode='json': datetime도 JSON 문자열로
print(m.model_dump(exclude={'where'}, mode='json'))

{'when': '2020-01-01T12:00:00', 'why': 'No idea'}


mode='json'은 json으로 출력
exclude 속성은 제외할 데이터 정하기

In [14]:
# model_dump_json(): JSON 문자열로 / exclude_defaults=True: 기본값과 같은 why 제외
print(m.model_dump_json(exclude_defaults=True))

{"when":"2020-01-01T12:00:00","where":"home"}


exclude_defaults=True는 기본값이랑 같은 속성들은 제외함

## 🔁 복습 질문

<details>
<summary>1. <code>signup_ts: datetime | None</code>인데 값을 안 넣으면 왜 오류가 나나?</summary>

기본값이 없으면 필수 필드다. `| None`은 "None도 넣을 수 있다"는 뜻일 뿐이다. 선택으로 만들려면 `= None`을 붙인다.
</details>

<details>
<summary>2. <code>color: Literal['red', 'green']</code>에 <code>'blue'</code>를 넣으면?</summary>

`ValidationError`. Literal은 적어 둔 값만 허용한다.
</details>

<details>
<summary>3. <code>exclude_unset=True</code>와 <code>exclude_defaults=True</code>의 차이는?</summary>

`exclude_unset`은 직접 넣지 않은 필드를, `exclude_defaults`는 값이 기본값과 같은 필드를 뺀다. 기본값과 같은 값을 직접 넣으면 앞의 것은 남기고 뒤의 것은 뺀다.
</details>

<details>
<summary>4. datetime이 들어 있는 모델을 JSON으로 보낼 수 있는 dict로 바꾸려면?</summary>

`model_dump(mode='json')`. datetime이 `'2020-01-01T12:00:00'` 같은 문자열로 바뀐다.
</details>
